# Customer Churn Prediction Using Machine Learning
**Course:** Artificial Intelligence  |  **Dataset:** Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn`)

**Goal:** predict whether a telecom customer will leave the company (`Churn = Yes`) or stay (`Churn = No`).

Pipeline: load data → clean → explore → encode features → split → train models → evaluate → predict on new customers.

## 0. Import libraries
We use **pandas/numpy** for data handling, **matplotlib/seaborn** for charts and **scikit-learn** for preprocessing, models and metrics.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

## 1. Data loading and understanding
We load the dataset with pandas. The assignment names an Excel file, but the public Kaggle copy is a CSV, so the code accepts either one.
The file must be in the same folder as this notebook.

In [ ]:
xlsx_path = "WA_Fn-UseC_-Telco-Customer-Churn.xlsx"
csv_path  = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

if os.path.exists(xlsx_path):
    df = pd.read_excel(xlsx_path)
elif os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError("Place the Telco dataset (.xlsx or .csv) in the same folder as this notebook.")

df.head()

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nColumn names:\n", df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True).round(3) * 100)

**Target variable:** `Churn` (Yes = customer left, No = customer stayed). Every other column is a possible input feature.

**What the important columns mean**

| Column | Meaning |
|---|---|
| `customerID` | Unique ID of the customer (identifier only) |
| `gender`, `SeniorCitizen`, `Partner`, `Dependents` | Basic customer profile |
| `tenure` | Number of months the customer has stayed with the company |
| `PhoneService`, `MultipleLines` | Whether the customer has phone service / more than one line |
| `InternetService` | Type of internet: DSL, Fiber optic or None |
| `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport` | Optional add-on services |
| `StreamingTV`, `StreamingMovies` | Streaming add-ons |
| `Contract` | Month-to-month, One year or Two year |
| `PaperlessBilling`, `PaymentMethod` | How the customer is billed and pays |
| `MonthlyCharges` | Amount charged every month |
| `TotalCharges` | Total amount charged over the whole stay |

## 2. Data preprocessing
We check for missing values, duplicates and wrong data types before modelling.

In [ ]:
print("Missing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())
print("\nTotalCharges dtype:", df["TotalCharges"].dtype)

`TotalCharges` should be a number, but it can be stored as text because a few entries are blank spaces. Models cannot use text, so we convert it to numeric.
Blank entries become `NaN`, and we then look at those rows.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing TotalCharges after conversion:", df["TotalCharges"].isnull().sum())
df[df["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

These rows all have `tenure = 0`: they are brand-new customers who have not been billed yet. So the correct value for `TotalCharges` is **0**, and we fill it with 0
instead of deleting the rows or using an average (which would be wrong for new customers).

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Drop duplicates if any (this dataset normally has none)
df = df.drop_duplicates()

# customerID is unique for every row, so it has no predictive value
df = df.drop(columns=["customerID"])

print("Missing values left:", df.isnull().sum().sum())
print("New shape:", df.shape)
df.dtypes

**Summary of cleaning:** converted `TotalCharges` to numeric, filled its 11 blanks with 0 (new customers), checked duplicates, and removed `customerID` because it is only an identifier.
`SeniorCitizen` is already 0/1 while other columns use Yes/No text; all of this is handled in the encoding step.

## 3. Exploratory Data Analysis (EDA)
### 3.1 Churn distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
counts = df["Churn"].value_counts()
sns.countplot(data=df, x="Churn", ax=ax[0], palette=["#4C9F70", "#D95F5F"])
ax[0].set_title("Churn count")
ax[1].pie(counts, labels=counts.index, autopct="%1.1f%%", colors=["#4C9F70", "#D95F5F"])
ax[1].set_title("Churn share")
plt.tight_layout(); plt.show()

### 3.2 Churn vs customer characteristics (categorical)
The table below shows the **churn rate (%)** inside each category. A big difference between categories means the column is useful for prediction.

In [ ]:
cat_features = ["gender", "SeniorCitizen", "Partner", "Dependents", "InternetService", "OnlineSecurity",
                "TechSupport", "Contract", "PaperlessBilling", "PaymentMethod"]
churn_flag = (df["Churn"] == "Yes").astype(int)

rate_table = {c: (churn_flag.groupby(df[c]).mean() * 100).round(1) for c in cat_features}
for c, t in rate_table.items():
    print(f"\nChurn rate (%) by {c}")
    print(t.to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, col in zip(axes.ravel(), ["Contract", "InternetService", "PaymentMethod", "TechSupport", "OnlineSecurity", "SeniorCitizen"]):
    sns.countplot(data=df, x=col, hue="Churn", ax=ax, palette=["#4C9F70", "#D95F5F"])
    ax.set_title(f"Churn by {col}")
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

### 3.3 Churn vs numeric characteristics

In [ ]:
df.groupby("Churn")[["tenure", "MonthlyCharges", "TotalCharges"]].mean().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.boxplot(data=df, x="Churn", y=col, ax=ax, palette=["#4C9F70", "#D95F5F"])
    ax.set_title(f"{col} vs Churn")
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, element="step")
plt.title("Tenure distribution by Churn"); plt.show()

### 3.4 Correlation with churn

In [ ]:
tmp = df.copy()
tmp["Churn"] = churn_flag
num_corr = tmp[["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Churn"]].corr()
plt.figure(figsize=(6, 4.5))
sns.heatmap(num_corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation of numeric features"); plt.show()

**Observations to confirm from your own output (write them in your own words in the report):**
- Roughly one in four customers churns, so the classes are **imbalanced**.
- **Month-to-month** contracts show a much higher churn rate than one- or two-year contracts.
- Churners have a much **lower tenure** and higher monthly charges.
- **Fiber optic** and **electronic check** customers churn more; customers with **TechSupport / OnlineSecurity** churn less.
- `gender` makes almost no difference.

## 4. Feature engineering and encoding
1. Convert the target `Churn` to 0/1.
2. Create one new feature: `AvgMonthlySpend` = TotalCharges / (tenure + 1), the average amount actually paid per month over the stay.
3. Convert all text columns to numbers with **one-hot encoding** (`get_dummies`). `drop_first=True` removes one redundant column per feature.
4. Separate the inputs **X** from the target **y**.

In [ ]:
data = df.copy()
data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})
data["AvgMonthlySpend"] = data["TotalCharges"] / (data["tenure"] + 1)

data_encoded = pd.get_dummies(data, drop_first=True).astype(float)

X = data_encoded.drop("Churn", axis=1)   # input features
y = data_encoded["Churn"].astype(int)    # target

print("X shape:", X.shape, "| y shape:", y.shape)
X.head()

## 5. Training and testing
- **Split:** 80% training, 20% testing. `stratify=y` keeps the same churn percentage in both parts.
- **Scaling:** Logistic Regression is sensitive to feature size, so numeric columns are standardized. The scaler is **fitted on the training data only**, to avoid data leakage. Tree-based models do not need scaling.
- **Models:** Logistic Regression (simple, interpretable baseline), Decision Tree (easy rules) and Random Forest (many trees, captures non-linear patterns).
- `class_weight="balanced"` makes the models pay more attention to the minority class (churners).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Churn share in train:", round(y_train.mean(), 3), "| in test:", round(y_test.mean(), 3))

num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "AvgMonthlySpend"]
scaler = StandardScaler()
X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols]  = scaler.transform(X_test[num_cols])

In [ ]:
models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE), True),
    "Decision Tree":       (DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=RANDOM_STATE), False),
    "Random Forest":       (RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced",
                                                   random_state=RANDOM_STATE, n_jobs=-1), False),
}

for name, (model, needs_scaling) in models.items():
    Xtr = X_train_s if needs_scaling else X_train
    model.fit(Xtr, y_train)
    print(f"Trained: {name}")

## 6. Model evaluation
Metrics (for the churn class = 1):
- **Accuracy:** share of all predictions that are correct.
- **Precision:** of the customers predicted to churn, how many really churned.
- **Recall:** of the customers who really churned, how many the model found.
- **F1-score:** balance between precision and recall.
- **Confusion matrix:** counts of true/false positives and negatives.

Because only about 26% of customers churn, accuracy alone can look good even for a weak model, so **recall and F1 matter more**.

In [ ]:
results, preds = [], {}
for name, (model, needs_scaling) in models.items():
    Xte = X_test_s if needs_scaling else X_test
    pred = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    preds[name] = pred
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-Score": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    })

results_df = pd.DataFrame(results).set_index("Model").round(3)
results_df

In [ ]:
for name in models:
    print(f"===== {name} =====")
    print(classification_report(y_test, preds[name], target_names=["No Churn", "Churn"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, name in zip(axes, models):
    cm = confusion_matrix(y_test, preds[name])
    ConfusionMatrixDisplay(cm, display_labels=["No Churn", "Churn"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
results_df[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", figsize=(10, 4.5), rot=0)
plt.title("Model comparison"); plt.ylim(0, 1); plt.legend(loc="lower right"); plt.show()

best_name = results_df["F1-Score"].idxmax()
print("Best model by F1-Score:", best_name)

**Comparison (write your own conclusion from the table above):** choose the model with the best balance of recall and F1 on the churn class, and explain *why* using the confusion matrices
(for example, how many real churners each model missed). Mention that accuracy alone is misleading with imbalanced classes.

In [ ]:
# Which features matter most? (uses the selected best model when it supports it)
best_model, best_scaled = models[best_name]
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=X.columns)
else:
    imp = pd.Series(best_model.coef_[0], index=X.columns)

top = imp.reindex(imp.abs().sort_values(ascending=False).index).head(10)
top.sort_values().plot(kind="barh", figsize=(8, 4.5), title=f"Top 10 influential features ({best_name})")
plt.show()

## 7. Application / Prediction on new customers
The function below takes a customer's information as a dictionary, applies the **same preprocessing** used for training (feature engineering, one-hot encoding, aligning columns, scaling)
and returns **Churn / No Churn** with a probability.

In [ ]:
def predict_churn(customer: dict, model_name=None):
    model_name = model_name or best_name
    model, needs_scaling = models[model_name]

    row = pd.DataFrame([customer])
    row["AvgMonthlySpend"] = row["TotalCharges"] / (row["tenure"] + 1)
    row = pd.get_dummies(row)
    row = row.reindex(columns=X.columns, fill_value=0).astype(float)   # same columns as training
    if needs_scaling:
        row[num_cols] = scaler.transform(row[num_cols])

    pred = int(model.predict(row)[0])
    prob = float(model.predict_proba(row)[0][1])
    return ("Churn" if pred == 1 else "No Churn"), round(prob, 3)

high_risk = {
    "gender": "Female", "SeniorCitizen": 0, "Partner": "No", "Dependents": "No", "tenure": 2,
    "PhoneService": "Yes", "MultipleLines": "No", "InternetService": "Fiber optic",
    "OnlineSecurity": "No", "OnlineBackup": "No", "DeviceProtection": "No", "TechSupport": "No",
    "StreamingTV": "No", "StreamingMovies": "No", "Contract": "Month-to-month",
    "PaperlessBilling": "Yes", "PaymentMethod": "Electronic check",
    "MonthlyCharges": 80.0, "TotalCharges": 160.0,
}
low_risk = {
    "gender": "Male", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "Yes", "tenure": 60,
    "PhoneService": "Yes", "MultipleLines": "Yes", "InternetService": "DSL",
    "OnlineSecurity": "Yes", "OnlineBackup": "Yes", "DeviceProtection": "Yes", "TechSupport": "Yes",
    "StreamingTV": "No", "StreamingMovies": "No", "Contract": "Two year",
    "PaperlessBilling": "No", "PaymentMethod": "Bank transfer (automatic)",
    "MonthlyCharges": 65.0, "TotalCharges": 3900.0,
}

print("Model used:", best_name)
for label, cust in [("High-risk customer", high_risk), ("Low-risk customer", low_risk)]:
    result, prob = predict_churn(cust)
    print(f"{label}: {result} (churn probability = {prob})")

Try changing the values in the dictionaries above (for example `Contract`, `tenure`, `MonthlyCharges`) and observe how the prediction changes.

In [ ]:
# Bonus: predictions for 5 real customers from the test set
sample = X_test.sample(5, random_state=1)
Xs = X_test_s.loc[sample.index] if best_scaled else sample
out = pd.DataFrame({
    "tenure": sample["tenure"],
    "MonthlyCharges": sample["MonthlyCharges"],
    "Actual": y_test.loc[sample.index].map({0: "No Churn", 1: "Churn"}),
    "Predicted": pd.Series(best_model.predict(Xs), index=sample.index).map({0: "No Churn", 1: "Churn"}),
    "Churn probability": best_model.predict_proba(Xs)[:, 1].round(3),
})
out

## 8. Conclusion
*(Complete with your own numbers from the outputs above.)*

- **Best model:** name the model with the best recall/F1 and explain why it is better than the others.
- **Data observations:** short tenure, month-to-month contracts, fiber optic internet, electronic-check payment and no tech support/security are linked with higher churn.
- **Limitations:** imbalanced classes, a fairly small dataset from a single quarter, no hyperparameter tuning or cross-validation, no data on customer complaints or competitor offers, and the model shows correlation, not cause.
- **Business use:** the company can score customers regularly, contact those with high churn probability, and offer discounts, contract upgrades or better support, spending the retention budget where it matters most.